# Document Prediction with AutoGluon

Document prediction classifies PDFs, scanned forms, invoices, receipts, and other document images. AutoGluon uses **LayoutLM**-family models that understand both the text content and the spatial layout of a document — not just the words, but where they appear on the page.

This matters because two documents can contain the same words but have completely different structure (e.g., an invoice vs. a contract).

This notebook covers:
1. System requirements (OCR engine, PDF renderer)
2. Data format: PDF/image file paths in a DataFrame
3. Training on a sample document dataset
4. The difference between digitally-created PDFs and scanned documents
5. Practical gotchas

## 0. System Dependency Check

Document prediction requires two system libraries:
1. **`poppler-utils`** — converts PDF pages to images for layout analysis
2. **`tesseract-ocr`** — extracts text from scanned/image-only documents

Both are pre-installed in the dev container Dockerfile. If you are running locally, install with:
```bash
# Ubuntu/Debian
sudo apt-get install -y poppler-utils tesseract-ocr

# macOS
brew install poppler tesseract
```

In [ ]:
import subprocess
import autogluon

print('AutoGluon version:', autogluon.__version__)

# Verify system dependencies are installed
for tool, cmd in [('poppler (pdfinfo)', ['pdfinfo', '-v']), ('tesseract', ['tesseract', '--version'])]:
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        print(f'✓ {tool} available')
    except FileNotFoundError:
        print(f'✗ {tool} NOT FOUND — install it before running this notebook')

## 1. Load the Sample Dataset

We use a subset of the **RVL-CDIP** dataset — a standard benchmark for document classification with 16 categories (letters, memos, emails, forms, handwritten, invoices, etc.).

In [ ]:
import os
import zipfile
import urllib.request
import pandas as pd

DATA_URL = 'https://autogluon.s3.amazonaws.com/datasets/rvl_cdip_sample.zip'
DATA_DIR = './rvl_cdip_sample'

if not os.path.exists(DATA_DIR):
    print('Downloading RVL-CDIP sample...')
    urllib.request.urlretrieve(DATA_URL, './rvl_cdip_sample.zip')
    with zipfile.ZipFile('./rvl_cdip_sample.zip', 'r') as z:
        z.extractall('.')
    os.remove('./rvl_cdip_sample.zip')
    print('Done.')

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f'Train: {train_df.shape} | Test: {test_df.shape}')
print('Label distribution:\n', train_df['label'].value_counts())
train_df.head()

In [ ]:
# Fix paths to be absolute
for df in [train_df, test_df]:
    df['doc_path'] = df['doc_path'].apply(
        lambda p: os.path.abspath(os.path.join(DATA_DIR, p)) if not os.path.isabs(p) else p
    )

# Validate a few paths exist
n_valid = train_df['doc_path'].apply(os.path.exists).sum()
print(f'{n_valid}/{len(train_df)} document files found')

# Check what kinds of documents we have
extensions = train_df['doc_path'].apply(lambda p: os.path.splitext(p)[1].lower()).value_counts()
print('File types:', extensions.to_dict())

In [ ]:
# Preview a few document pages
import matplotlib.pyplot as plt
from PIL import Image

sample = train_df.groupby('label').first().reset_index().head(4)

fig, axes = plt.subplots(1, len(sample), figsize=(14, 5))
for ax, (_, row) in zip(axes, sample.iterrows()):
    fpath = row['doc_path']
    ext = os.path.splitext(fpath)[1].lower()
    if ext == '.pdf':
        # Convert first PDF page to image for display
        import subprocess, tempfile
        with tempfile.NamedTemporaryFile(suffix='.png') as tmp:
            subprocess.run(['pdftoppm', '-png', '-singlefile', '-r', '72', fpath, tmp.name[:-4]], check=True)
            img = Image.open(tmp.name)
    else:
        img = Image.open(fpath)

    ax.imshow(img, cmap='gray')
    ax.set_title(row['label'], fontsize=9)
    ax.axis('off')

plt.suptitle('Sample Documents by Category', fontsize=11)
plt.tight_layout()
plt.show()

## 2. Digitally-Created vs. Scanned PDFs

There are two fundamentally different types of PDFs:

| Type | Description | OCR needed? | Quality |
|------|-------------|-------------|--------|
| **Digitally created** | Text is stored as actual characters; can be copy-pasted | No | High |
| **Scanned** | The document is a photo/image; text must be extracted via OCR | Yes | Depends on scan quality |

AutoGluon handles both, but scanned documents will have lower accuracy if OCR quality is poor (blurry scans, handwriting, unusual fonts).

In [ ]:
# Check if a PDF has embedded text (digitally created) or is image-only (scanned)
import subprocess

def check_pdf_type(pdf_path: str) -> str:
    """Returns 'digital' if the PDF has extractable text, 'scanned' if image-only."""
    result = subprocess.run(
        ['pdftotext', pdf_path, '-'],
        capture_output=True, text=True
    )
    return 'digital' if result.stdout.strip() else 'scanned'

pdf_files = train_df[train_df['doc_path'].str.endswith('.pdf')]['doc_path'].head(5)
for path in pdf_files:
    print(f'{os.path.basename(path)}: {check_pdf_type(path)}')

## 3. Train the Document Classifier

In [ ]:
from autogluon.multimodal import MultiModalPredictor

predictor = MultiModalPredictor(
    label='label',
    problem_type='multiclass',
    path='./ag_document_model',
)

predictor.fit(
    train_data=train_df,
    time_limit=300,
    hyperparameters={
        # LayoutLMv3 understands document layout (text position) in addition to content
        'model.document_transformer.checkpoint_name': 'microsoft/layoutlmv3-base',
    },
)

In [ ]:
y_pred = predictor.predict(test_df)
metrics = predictor.evaluate(test_df)
print('Test metrics:', metrics)

# Show a few predictions
comparison = pd.DataFrame({
    'file':  [os.path.basename(p) for p in test_df['doc_path'].head(5)],
    'true':  test_df['label'].head(5).values,
    'pred':  y_pred[:5].values,
})
comparison['correct'] = comparison['true'] == comparison['pred']
print(comparison.to_string(index=False))

## 4. Predicting on a Single New Document

In [ ]:
# Create a DataFrame with just one document to classify
single_doc = pd.DataFrame({'doc_path': [test_df['doc_path'].iloc[0]]})

pred = predictor.predict(single_doc)
prob = predictor.predict_proba(single_doc)

print(f'Predicted class: {pred[0]}')
print('\nTop 3 probabilities:')
top3 = prob.iloc[0].sort_values(ascending=False).head(3)
for cls, p in top3.items():
    print(f'  {cls}: {p:.2%}')

## ⚠️ Practical Gotchas

| Gotcha | What Happens | Fix |
|--------|-------------|-----|
| **`poppler-utils` not installed** | `PDFInfoNotInstalledError` or `pdfinfo: command not found` | `sudo apt-get install poppler-utils` (pre-installed in dev container) |
| **`tesseract-ocr` not installed** | OCR falls back silently or errors | `sudo apt-get install tesseract-ocr` (pre-installed in dev container) |
| **Scanned PDFs with poor quality** | OCR extracts garbage text → model gets bad input | Pre-process scans with deskewing, denoising, or upscaling before fitting |
| **Password-protected PDFs** | Decryption error (poppler cannot open) | Decrypt with `qpdf --decrypt input.pdf output.pdf` before training |
| **Very long PDFs (50+ pages)** | Slow processing; may OOM | Truncate to first N pages: `pdfselect` or use PyPDF2 |
| **PDFs with unusual encodings** | Text extraction fails or returns mojibake | Try `pdftotext -enc UTF-8` first; if it fails, fall back to OCR mode |
| **AutoGluon uses first page only** | Multi-page PDFs: only page 1 is used for classification | If classification depends on later pages, extract the relevant page into a separate PDF |
| **Column detection with `.pdf` paths** | Automatically detected as `document` modality | If paths are passed but not detected correctly, use `column_types={'doc_path': 'document'}` |